In [0]:
from pyspark.sql import functions as F, Window

B = "olist_proyecto.bronce"
P = "olist_proyecto.plata"

In [0]:
o = spark.table(f"{B}.orders")

o = (o.select(
        "order_id", "customer_id", "order_status",
        F.to_timestamp("order_purchase_timestamp").alias("fecha_compra"),
        F.to_timestamp("order_approved_at").alias("fecha_aprobacion"),
        F.to_timestamp("order_delivered_carrier_date").alias("fecha_envio"),
        F.to_timestamp("order_delivered_customer_date").alias("fecha_entrega"),
        F.to_timestamp("order_estimated_delivery_date").alias("fecha_estimada")))

dudosa = (
    (F.col("fecha_envio") < F.col("fecha_compra")) |
    (F.col("fecha_entrega") < F.col("fecha_envio")) |
    ((F.col("order_status") == "delivered") & F.col("fecha_entrega").isNull())
)

o = o.withColumn("fecha_dudosa", F.when(dudosa, "si").otherwise("no"))
o.write.mode("overwrite").saveAsTable(f"{P}.orders")

In [0]:
r = spark.table(f"{B}.order_reviews")

r = (r.select("review_id", "order_id", "review_score",
              "review_comment_title", "review_comment_message",
              F.to_timestamp("review_creation_date").alias("fecha_reseña"),
              F.to_timestamp("review_answer_timestamp").alias("fecha_respuesta"))
       .withColumn("review_score", F.col("review_score").cast("int")))

w = Window.partitionBy("order_id").orderBy(F.col("fecha_reseña").desc(),
                                           F.col("fecha_respuesta").desc())
r = r.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
r.write.mode("overwrite").saveAsTable(f"{P}.order_reviews")

In [0]:
p = spark.table(f"{B}.products").drop("archivo_origen", "fecha_carga")
p = p.fillna({"product_category_name": "sin_categoria"})
p.write.mode("overwrite").saveAsTable(f"{P}.products")

In [0]:
oi = (spark.table(f"{B}.order_items")
      .select("order_id", "order_item_id", "product_id", "seller_id",
              F.to_timestamp("shipping_limit_date").alias("fecha_limite_envio"),
              F.col("price").cast("double"),
              F.col("freight_value").cast("double")))
oi.write.mode("overwrite").saveAsTable(f"{P}.order_items")

op = (spark.table(f"{B}.order_payments")
      .select("order_id",
              F.col("payment_sequential").cast("int"),
              "payment_type",
              F.col("payment_installments").cast("int"),
              F.col("payment_value").cast("double")))
op.write.mode("overwrite").saveAsTable(f"{P}.order_payments")

In [0]:
g = (spark.table(f"{B}.geolocation")
     .select("geolocation_zip_code_prefix",
             F.col("geolocation_lat").cast("double").alias("lat"),
             F.col("geolocation_lng").cast("double").alias("lng"),
             "geolocation_city", "geolocation_state")
     .dropDuplicates())

g = g.filter(F.col("lat").between(-34, 6) & F.col("lng").between(-74, -34))
g.write.mode("overwrite").saveAsTable(f"{P}.geolocation")

In [0]:
for t in ["customers", "sellers", "category_translation"]:
    (spark.table(f"{B}.{t}").drop("archivo_origen", "fecha_carga")
          .write.mode("overwrite").saveAsTable(f"{P}.{t}"))

In [0]:
spark.sql(f"SELECT COUNT(*) total, SUM(CASE WHEN fecha_dudosa='si' THEN 1 ELSE 0 END) dudosas FROM {P}.orders").show()
spark.sql(f"SELECT COUNT(*) filas, COUNT(DISTINCT order_id) pedidos FROM {P}.order_reviews").show()
spark.sql(f"SELECT COUNT(*) sin_cat FROM {P}.products WHERE product_category_name='sin_categoria'").show()

In [0]:
print("geolocation:", spark.table("olist_proyecto.plata.geolocation").count())
spark.sql("SHOW TABLES IN olist_proyecto.plata").show()

In [0]:
g = spark.table("olist_proyecto.bronce.geolocation").select(
    F.col("geolocation_lat").cast("double").alias("lat"),
    F.col("geolocation_lng").cast("double").alias("lng"))

fuera = ~(F.col("lat").between(-34, 6) & F.col("lng").between(-74, -34))
print("fuera de Brasil (con duplicados):", g.filter(fuera).count())

In [0]:
g = spark.table("olist_proyecto.bronce.geolocation").select(
    "geolocation_zip_code_prefix",
    F.col("geolocation_lat").cast("double").alias("lat"),
    F.col("geolocation_lng").cast("double").alias("lng"),
    "geolocation_city", "geolocation_state")

fuera = ~(F.col("lat").between(-34, 6) & F.col("lng").between(-74, -34))
excl = g.filter(fuera).dropDuplicates()

print("puntos distintos excluidos:", excl.count())
display(excl.orderBy("lng"))
